In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score

# --- 1. THE AUGMENTED DATA FETCH ---
print("Fetching Augmented Data (Injecting India VIX)...")
tickers = {
    'Nifty': '^NSEI',
    'USD_INR': 'INR=X',
    'Crude_Oil': 'CL=F',
    'Gold': 'GC=F',
    'US_10Y': '^TNX',
    'India_VIX': '^INDIAVIX'  # <-- THE NEW DOMAIN KNOWLEDGE
}

data = yf.download(list(tickers.values()), period='5y', interval='1d', progress=False)['Close']
data.columns = list(tickers.keys())
data.ffill(inplace=True)

# --- 2. FEATURE ENGINEERING ---
def engineer_features(df, column_name):
    # 20-day Volatility
    df[f'{column_name}_Vol_20d'] = df[column_name].pct_change().rolling(window=20).std()

    # 10-day and 20-day Z-Scores (The timeframes we proved were the best)
    for w in [10, 20]:
        rolling_mean = df[column_name].rolling(window=w).mean()
        rolling_std = df[column_name].rolling(window=w).std() + 1e-8
        df[f'{column_name}_ZScore_{w}d'] = (df[column_name] - rolling_mean) / rolling_std
    return df

macro_variables = ['USD_INR', 'Crude_Oil', 'Gold', 'US_10Y', 'India_VIX']
for macro in macro_variables:
    data = engineer_features(data, macro)

data.dropna(inplace=True)

# --- 3. THE TARGET ---
lookahead = 5
data['Future_5d_Ret'] = data['Nifty'].pct_change(lookahead).shift(-lookahead)
data['Future_Absolute_Move'] = data['Future_5d_Ret'].abs()
volatility_threshold = data['Future_Absolute_Move'].quantile(0.75)
data['Target_High_Vol'] = np.where(data['Future_Absolute_Move'] > volatility_threshold, 1, 0)
data.dropna(inplace=True)

# --- 4. THE ULTIMATE PRUNED FEATURE SET ---
# We keep our elite features from Version 3, plus the new VIX indicators
keepers = [
    'US_10Y', 'US_10Y_Vol_20d', 'Crude_Oil_ZScore_20d',
    'US_10Y_ZScore_20d', 'Gold', 'Gold_Vol_20d',
    'Gold_ZScore_20d', 'USD_INR', 'USD_INR_Vol_20d',
    'Crude_Oil_Vol_20d', 'USD_INR_ZScore_10d',
    'India_VIX', 'India_VIX_Vol_20d', 'India_VIX_ZScore_10d', 'India_VIX_ZScore_20d' # <-- New additions
]

X = data[keepers]
y = data['Target_High_Vol']

# --- 5. WALK-FORWARD SNIPER ENGINE ---
print("\nInitializing Version 4.0 Walk-Forward Engine...")
tscv = TimeSeriesSplit(n_splits=5)

xgb_model = xgb.XGBClassifier(
    n_estimators=100, learning_rate=0.05, max_depth=3,
    scale_pos_weight=3, random_state=42, eval_metric='logloss'
)

fold_accuracies, fold_precisions, fold_recalls = [], [], []
fold = 1

for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    xgb_model.fit(X_train, y_train)

    y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]
    y_pred_custom = (y_pred_proba >= 0.70).astype(int) # 70% Sniper Threshold

    acc = accuracy_score(y_test, y_pred_custom)
    prec = precision_score(y_test, y_pred_custom, zero_division=0)
    rec = recall_score(y_test, y_pred_custom, zero_division=0)

    fold_accuracies.append(acc)
    fold_precisions.append(prec)
    fold_recalls.append(rec)
    print(f"Fold {fold} | Accuracy: {acc:.2%} | Precision: {prec:.2%} | Recall: {rec:.2%}")
    fold += 1

print("\n" + "="*40)
print(f"Final Average Accuracy:  {np.mean(fold_accuracies):.2%}")
print(f"Final Average Precision: {np.mean(fold_precisions):.2%}")
print(f"Final Average Recall:    {np.mean(fold_recalls):.2%}")
print("="*40)

Fetching Augmented Data (Injecting India VIX)...


/tmp/ipykernel_903/1234189907.py:19: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(list(tickers.values()), period='5y', interval='1d', progress=False)['Close']



Initializing Version 4.0 Walk-Forward Engine...
Fold 1 | Accuracy: 59.62% | Precision: 0.00% | Recall: 0.00%
Fold 2 | Accuracy: 64.79% | Precision: 28.00% | Recall: 10.94%
Fold 3 | Accuracy: 79.81% | Precision: 18.75% | Recall: 9.09%
Fold 4 | Accuracy: 77.00% | Precision: 0.00% | Recall: 0.00%
Fold 5 | Accuracy: 78.87% | Precision: 66.67% | Recall: 4.35%

Final Average Accuracy:  72.02%
Final Average Precision: 22.68%
Final Average Recall:    4.88%
